# Auto-Ingestion: From Database to Semantic Layer

SLayer can introspect a database schema and automatically generate models with dimensions, measures, and join relationships. This notebook shows what happens under the hood.

We use the **Jaffle Shop** dataset — a synthetic e-commerce schema with 7 tables and foreign key relationships between them.

See also: [Auto-Ingestion docs](../../concepts/ingestion.md) | [Models docs](../../concepts/models.md)

**Prerequisites:** `pip install motley-slayer[examples]`

In [1]:
import os
import sys

import duckdb
import sqlalchemy as sa

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop, DB_PATH

engine, storage, models = ensure_jaffle_shop()

/home/james/miniconda3/envs/motley3.11/lib/python3.11/site-packages/sqlglot/tokens.py:14: UserWarning: sqlglot[rs] is deprecated and no longer compatible with sqlglot. Please use sqlglotc instead for faster parsing: pip install sqlglot[c]
  warnings.warn(


## The Jaffle Shop Schema

The database has 7 tables connected by foreign keys:

```
customers  <──  orders  ──>  stores
                  │
            order_items  ──>  products  <──  supplies

customers  <──  tweets
```

Let's peek at the data:

In [2]:
conn = duckdb.connect(DB_PATH, read_only=True)
for table in ["customers", "stores", "products", "orders", "order_items", "supplies", "tweets"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:>15}: {count:>8,} rows")
conn.close()

      customers:    1,937 rows
         stores:        6 rows
       products:       10 rows
         orders:  206,385 rows
    order_items:  324,415 rows
       supplies:       65 rows
         tweets:   96,663 rows


## Step 1: FK Graph Discovery

The first thing auto-ingestion does is introspect foreign key constraints and build a **directed dependency graph**. Each edge means "this table has a FK pointing to that table."

The graph must be acyclic — if cycles are found, SLayer raises a `RollupGraphError`.

For each table, SLayer then computes the **transitive closure**: all tables reachable by following FK chains. This determines which joined dimensions will be available.

In [3]:
from slayer.core.models import DatasourceConfig
from slayer.engine.ingestion import _build_fk_graph, _compute_transitive_closure

ds = DatasourceConfig(name="jaffle_shop", type="duckdb", database=DB_PATH)
sa_engine = sa.create_engine(ds.resolve_env_vars().get_connection_string())
inspector = sa.inspect(sa_engine)
table_names = inspector.get_table_names()

fk_graph = _build_fk_graph(inspector=inspector, table_names=table_names, schema=None)

print("FK Graph (table -> tables it references):")
for table in sorted(fk_graph):
    refs = sorted(fk_graph[table])
    print(f"  {table} -> {refs}")

print()
print("Tables with no FK references (leaf tables):")
for t in sorted(table_names):
    if t not in fk_graph:
        print(f"  {t}")

FK Graph (table -> tables it references):
  order_items -> ['orders', 'products']
  orders -> ['customers', 'stores']
  supplies -> ['products']
  tweets -> ['customers']

Tables with no FK references (leaf tables):
  customer_segments
  customers
  products
  stores


In [4]:
# Transitive closure: what tables are reachable from order_items?
for table in ["orders", "order_items"]:
    reachable = _compute_transitive_closure(fk_graph, table)
    print(f"{table} can reach: {sorted(reachable)}")

sa_engine.dispose()

orders can reach: ['customers', 'stores']
order_items can reach: ['customers', 'orders', 'products', 'stores']


Notice that `order_items` transitively reaches `customers` and `stores` through `orders`, even though it has no direct FK to them.

## Step 2: Join Generation

SLayer converts FK relationships into `ModelJoin` objects via BFS. Each join specifies:
- **target_model**: the table being joined
- **join_pairs**: `[[source_column, target_column]]` — the columns to join on

For multi-hop joins (e.g., `order_items -> orders -> customers`), the source column is path-qualified: `"orders.customer_id"` means "the `customer_id` column in the already-joined `orders` table."

In [5]:
orders_model = next(m for m in models if m.name == "orders")
oi_model = next(m for m in models if m.name == "order_items")

print("=== orders model joins (2 direct FKs) ===")
for j in orders_model.joins:
    print(f"  -> {j.target_model}  ON {j.join_pairs}")

print()
print("=== order_items model joins (2 direct + 2 multi-hop) ===")
for j in oi_model.joins:
    src = j.join_pairs[0][0]
    tgt = j.join_pairs[0][1]
    multi_hop = " (multi-hop)" if "." in src else ""
    print(f"  -> {j.target_model}  ON {src} = {tgt}{multi_hop}")

=== orders model joins (2 direct FKs) ===
  -> customers  ON [['customer_id', 'id']]
  -> stores  ON [['store_id', 'id']]

=== order_items model joins (2 direct + 2 multi-hop) ===
  -> orders  ON order_id = id
  -> products  ON sku = sku
  -> customers  ON orders.customer_id = id (multi-hop)
  -> stores  ON orders.store_id = id (multi-hop)


## Step 3: Dimension & Measure Generation

For each table, SLayer generates:
- **Dimensions** for every column (own + joined). Joined dimensions use dot syntax in their `name` (e.g., `customers.name`) and `__`-delimited aliases in their `sql` (e.g., `customers.name` for SQL).
- **Measures**: `count` always; numeric non-ID columns get `_sum`, `_avg`, `_min`, `_max`, `_distinct`; non-numeric get `_distinct` and `_count`; joined table PKs get a `tablename.count` (COUNT DISTINCT).

In [6]:
print("=== orders model dimensions ===")
print(f"{'Name':<25} {'SQL':<25} {'Type':<8} {'Source'}")
print("-" * 75)
for dim in orders_model.dimensions:
    source = "own" if "." not in dim.name else f"joined ({dim.description})"
    pk = " [PK]" if dim.primary_key else ""
    print(f"{dim.name:<25} {dim.sql:<25} {dim.type:<8} {source}{pk}")

=== orders model dimensions ===
Name                      SQL                       Type     Source
---------------------------------------------------------------------------
id                        id                        string   own [PK]
customer_id               customer_id               string   own
ordered_at                ordered_at                date     own
store_id                  store_id                  string   own
subtotal                  subtotal                  number   own
tax_paid                  tax_paid                  number   own
order_total               order_total               number   own
customers.id              customers.id              string   joined (From customers)
customers.name            customers.name            string   joined (From customers)
stores.id                 stores.id                 string   joined (From stores)
stores.name               stores.name               string   joined (From stores)
stores.opened_at          stor

In [7]:
# Multi-hop dimensions from order_items show the __ alias convention
print("=== order_items: multi-hop dimensions (via orders) ===")
print(f"{'Dimension name (dots)':<30} {'SQL expression (__)':<30}")
print("-" * 62)
for dim in oi_model.dimensions:
    if dim.name.startswith("orders.") and "." in dim.name[len("orders."):]:
        print(f"{dim.name:<30} {dim.sql:<30}")

=== order_items: multi-hop dimensions (via orders) ===
Dimension name (dots)          SQL expression (__)           
--------------------------------------------------------------
orders.customers.id            orders__customers.id          
orders.customers.name          orders__customers.name        
orders.stores.id               orders__stores.id             
orders.stores.name             orders__stores.name           
orders.stores.opened_at        orders__stores.opened_at      
orders.stores.tax_rate         orders__stores.tax_rate       


In [8]:
print("=== orders model measures (selected) ===")
print(f"{'Name':<30} {'Type':<16} {'SQL':<20} {'Notes'}")
print("-" * 80)
for m in orders_model.measures:
    notes = ""
    if m.name == "count":
        notes = "always generated"
    elif m.name.endswith(".count"):
        notes = m.description or ""
    elif "_sum" in m.name or "_avg" in m.name:
        notes = "numeric column"
    elif "_distinct" in m.name or "_count" in m.name:
        notes = "non-numeric or distinct"
    sql_str = m.sql or "(COUNT(*))"
    print(f"{m.name:<30} {m.type:<16} {sql_str:<20} {notes}")

=== orders model measures (selected) ===
Name                           Type             SQL                  Notes
--------------------------------------------------------------------------------
count                          count            (COUNT(*))           always generated
subtotal_sum                   sum              subtotal             numeric column
subtotal_avg                   avg              subtotal             numeric column
subtotal_min                   min              subtotal             
subtotal_max                   max              subtotal             
subtotal_distinct              count_distinct   subtotal             non-numeric or distinct
tax_paid_sum                   sum              tax_paid             numeric column
tax_paid_avg                   avg              tax_paid             numeric column
tax_paid_min                   min              tax_paid             
tax_paid_max                   max              tax_paid             
tax_paid

## Summary

Auto-ingestion discovers the database structure and generates a complete semantic layer:

1. **FK graph** identifies table relationships and validates they're acyclic
2. **Transitive closure** determines which tables should be joined (including multi-hop)
3. **Join metadata** (`ModelJoin` objects) captures how to join tables at query time
4. **Dimensions and measures** are generated with proper naming conventions

No join-related SQL is baked into the models — JOINs are constructed dynamically at query time from the join metadata.

**Next:** See the [Example Queries](../example_queries/example_queries.ipynb) notebook to query this data, or the [Joins](../joins/joins.ipynb) notebook for a deep dive into join mechanics.